In [0]:
# Databricks notebook source
import time
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               LongType, TimestampType, DoubleType)

# --- Parâmetros (o Job sobrescreve estes valores) ---
dbutils.widgets.text("catalog", "olist")
dbutils.widgets.text("raw_path", "/Volumes/olist/raw/files")
dbutils.widgets.text("run_id", "manual")

CATALOG  = dbutils.widgets.get("catalog")
RAW_PATH = dbutils.widgets.get("raw_path")
RUN_ID   = dbutils.widgets.get("run_id")

# COMMAND ----------

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.ops")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.ops.ingestion_log (
    run_id STRING, layer STRING, table_name STRING, source_file STRING,
    status STRING, row_count BIGINT,
    started_at TIMESTAMP, finished_at TIMESTAMP, duration_s DOUBLE,
    error STRING
)""")

tables = {
    "customers":            "olist_customers_dataset.csv",
    "geolocation":          "olist_geolocation_dataset.csv",
    "order_items":          "olist_order_items_dataset.csv",
    "order_payments":       "olist_order_payments_dataset.csv",
    "order_reviews":        "olist_order_reviews_dataset.csv",
    "orders":               "olist_orders_dataset.csv",
    "products":             "olist_products_dataset.csv",
    "sellers":              "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

# COMMAND ----------

logs = []
for name, file in tables.items():
    started = datetime.now(timezone.utc)
    t0 = time.time()
    target = f"{CATALOG}.bronze.{name}"
    try:
        df = (spark.read
              .option("header", True)
              .option("multiLine", True)
              .option("escape", '"')
              .csv(f"{RAW_PATH}/{file}")
              .withColumn("_source_file", F.col("_metadata.file_path"))
              .withColumn("_ingested_at", F.current_timestamp())
              .withColumn("_run_id", F.lit(RUN_ID)))
        (df.write.mode("overwrite")
           .option("overwriteSchema", True)
           .saveAsTable(target))
        rows, status, error = spark.table(target).count(), "SUCCESS", None
    except Exception as e:
        rows, status, error = None, "FAILED", str(e)[:2000]

    logs.append((RUN_ID, "bronze", name, file, status, rows,
                 started, datetime.now(timezone.utc),
                 round(time.time() - t0, 2), error))
    print(f"[{status}] {name}: {rows} linhas")

# COMMAND ----------

log_schema = StructType([
    StructField("run_id", StringType()), StructField("layer", StringType()),
    StructField("table_name", StringType()), StructField("source_file", StringType()),
    StructField("status", StringType()), StructField("row_count", LongType()),
    StructField("started_at", TimestampType()), StructField("finished_at", TimestampType()),
    StructField("duration_s", DoubleType()), StructField("error", StringType()),
])
spark.createDataFrame(logs, log_schema) \
     .write.mode("append").saveAsTable(f"{CATALOG}.ops.ingestion_log")

# Falha o Job se alguma tabela falhou, para disparar a notificação
failed = [l[2] for l in logs if l[4] == "FAILED"]
if failed:
    raise Exception(f"Falha na ingestão de: {failed}")